# Parses Data from a Site and Extracts Tables

In [ ]:
import re
import httpx
import pandas as pd
from typing import List, Optional
#import io

from itables import init_notebook_mode  # For interactive pdf dataframe viewing 
init_notebook_mode(all_interactive=True)

## Utility Functions

In [ ]:
def fetch_site(url: str, timeout: float = 10.0, headers: Optional[dict] = None) -> str:
    """
    Fetch the raw HTML content of a site using httpx.

    Args:
        url: The URL to request.
        timeout: Request timeout in seconds.
        headers: Optional custom headers (defaults to a basic User-Agent).

    Returns:
        The response body as a string.
    """
    default_headers = {
        "User-Agent": "Mozilla/5.0 (compatible; TableScraper/1.0)"
    }
    if headers:
        default_headers.update(headers)

    with httpx.Client(timeout=timeout, follow_redirects=True) as client:
        response = client.get(url, headers=default_headers)
        response.raise_for_status()
        return response.text

In [ ]:
def extract_tables(html: str) -> List[pd.DataFrame]:
    """
    Extract HTML tables (with multiple rows/columns) from raw HTML using regex,
    and return them as a list of pandas DataFrames.

    Args:
        html: Raw HTML content as a string.

    Returns:
        List of pandas.DataFrame objects, one per detected table.
    """
    tables_html = re.findall(
        r"<table[^>]*>(.*?)</table>", html, flags=re.DOTALL | re.IGNORECASE
    )

    def clean_cell(cell: str) -> str:
        # Strip any nested tags and collapse whitespace
        text = re.sub(r"<[^>]+>", " ", cell)
        text = re.sub(r"&nbsp;", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    dataframes = []

    for table_html in tables_html:
        # Extract each row
        rows_html = re.findall(
            r"<tr[^>]*>(.*?)</tr>", table_html, flags=re.DOTALL | re.IGNORECASE
        )

        rows_data = []
        for row_html in rows_html:
            # Extract cells (th or td)
            cells = re.findall(
                r"<t[hd][^>]*>(.*?)</t[hd]>", row_html, flags=re.DOTALL | re.IGNORECASE
            )
            cleaned = [clean_cell(c) for c in cells]
            if cleaned:
                rows_data.append(cleaned)

        if len(rows_data) < 2:
            # Skip tables that don't have at least a header + one data row
            continue

        # Normalize row lengths (pad shorter rows with empty strings)
        max_cols = max(len(r) for r in rows_data)
        rows_data = [r + [""] * (max_cols - len(r)) for r in rows_data]

        header, *body = rows_data
        df = pd.DataFrame(body, columns=header if len(set(header)) == len(header) else None)
        dataframes.append(df)

    return dataframes

## Extract

In [ ]:
url_idx = 0

urls = [
    "https://finance.yahoo.com/quote/%5ENDX/history/",
    "https://finance.yahoo.com/quote/%5EDJI/history/",
    "https://finance.yahoo.com/quote/GOOG/history/",
]

# Test:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"

url = urls[url_idx]
print(f"{url}")

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
}

html_content = fetch_site(url, headers=headers)

In [ ]:
tables = extract_tables(html_content)

print(f"Found {len(tables)} table(s).")
for i, df in enumerate(tables):
    print(f"\n--- Table {i + 1} ---")
    # print(df)
    display(df.describe())
    display(df.head(5))

In [ ]:
df = tables[0].copy()

# Fix date format:
df["Date"] = pd.to_datetime(df["Date"])
df["Date"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")

# # Drop unwanted columns
# df = df.drop(df.columns[-2], axis=1)
# df = df.drop(df.columns[-2], axis=1)

# Rename Columns:
df = df.rename(columns={
    df.columns[-3]: "Close",
    df.columns[-2]: "Adj Close",
    })

# Convert to numerical:
for col_name in ["Open", "High", "Low", "Close", "Adj Close", "Volume"]:
    # print(col_name)
    df[col_name] = df[col_name].str.replace(",", "")
    df[col_name] = pd.to_numeric(df[col_name], errors="coerce")

# Reverse the rows:
df = df[::-1]

display(df.describe())
display(df)

## Append to CSV:

In [ ]:
print(f"{url=}")
csv_file = [
    "./csv_files/NDX.csv",
    "./csv_files/DJI.csv",
    "./csv_files/GOOG.csv",
][url_idx]
print(f"{csv_file=}")

In [ ]:
# Read CSV:
csv_df = pd.read_csv(csv_file)
print(f"{len(csv_df.index)=}")
print(f"{len(df.index)=}")
df = df.dropna()
print(f"{len(df.index)=}")

# Convert "Date" to index
csv_df.set_index("Date", inplace=True)
df.set_index("Date", inplace=True)

# # Use Update
# csv_df.update(df)

# Use concat
csv_df = pd.concat(
    [csv_df[~csv_df.index.isin(df.index)], df]
)

# Convert back "Date" to Column:
csv_df.reset_index(inplace=True)
df.reset_index(inplace=True)

# Sort:
# Use the ascending=False parameter
csv_df = csv_df.sort_values(
    by="Date", ascending=False, ignore_index=True,
)

print(f"{len(csv_df.index)=}")
display(csv_df.describe())
display(csv_df)

In [ ]:
csv_df.to_csv(csv_file, index=False)